# **Sistem Pakar Penyakit Tanaman Jahe Menggunakan Metode Forward Chaining**

# 1. Pendahuluan

Sistem pakar merupakan salah satu cabang kecerdasan buatan yang dirancang untuk merepresentasikan pengetahuan dan proses pengambilan keputusan seorang pakar ke dalam sistem komputer. Sistem ini bekerja dengan memanfaatkan basis pengetahuan yang terdiri atas fakta dan aturan, serta mesin inferensi yang berfungsi untuk menarik kesimpulan berdasarkan informasi yang tersedia. Dalam implementasinya, sistem pakar banyak digunakan pada domain yang membutuhkan diagnosis, prediksi, dan rekomendasi berbasis pengetahuan ahli.

Salah satu pendekatan utama dalam sistem pakar adalah metode **forward chaining**. Metode ini bertujuan untuk menghasilkan kesimpulan berdasarkan sekumpulan fakta dan aturan yang terstruktur. Metode ini dapat digunakan untuk menentukan pola penyakit tanaman yang paling mungkin menyerang tanaman tersebut.

# 2. Sistem Pakar Tanaman Jahe

Dalam konteks pertanian, khususnya pada budidaya tanaman jahe, identifikasi penyakit secara dini merupakan faktor kritis yang mempengaruhi produktivitas dan kualitas hasil panen. Namun, keterbatasan akses terhadap pakar tanaman sering menjadi kendala bagi petani dalam melakukan diagnosis yang akurat. Oleh karena itu, diperlukan suatu sistem berbasis pengetahuan yang mampu membantu proses identifikasi penyakit berdasarkan gejala yang diamati.

Kasus yang dikaji dalam notebook ini berfokus pada diagnosis penyakit tanaman jahe berdasarkan gejala berupa daun menguning dan munculnya eksudat pada batang. Sistem akan memanfaatkan sekumpulan aturan produksi yang merepresentasikan hubungan antara gejala dan kondisi penyakit. Dengan menggunakan metode forward chaining, sistem akan melakukan inferensi secara bertahap untuk menemukan jenis hama atau penyakit yang paling mungkin menyerang tanaman tersebut.

Tujuan dari implementasi ini adalah untuk membangun model sederhana sistem pakar berbasis aturan yang mampu menjelaskan alur penalaran secara transparan. Selain itu, sistem ini juga berfungsi sebagai media pembelajaran untuk memahami bagaimana mesin inferensi bekerja dalam menghasilkan kesimpulan dari sekumpulan fakta dan aturan yang terstruktur.


### **1. Definisi rule dan verifikasi rule base**

In [ ]:
# Definisi rule
rules = [
    {"id": "R1", "if": ["tanaman jahe terkena penyakit layu bakteri"], "then": "R. solanacearum"},
    {"id": "R2", "if": ["batang membusuk", "terlihat luka pada rimpang"], "then": "Nematoda"},
    {"id": "R3", "if": ["muncul eksudat di rimpang"], "then": "muncul gejala water soak"},
    {"id": "R4", "if": ["batang membusuk"], "then": "batang terkelupas"},
    {"id": "R5", "if": ["daun jahe menguning"], "then": "daun membusuk"},
    {"id": "R6", "if": ["daun membusuk", "batang membusuk"], "then": "akar membusuk"},
    {"id": "R7", "if": ["batang terkelupas"], "then": "akar terkelupas"},
    {"id": "R8", "if": ["daun jahe menguning"], "then": "tanaman tumbuh kerdil"},
    {"id": "R9", "if": ["akar membusuk"], "then": "muncul eksudat di batang"},
    {"id": "R10", "if": ["daun membusuk"], "then": "batang membusuk"},
    {"id": "R11", "if": ["akar terkelupas"], "then": "muncul luka di rimpang"},
    {"id": "R12", "if": ["muncul eksudat di batang"], "then": "muncul eksudat di rimpang"},
    {"id": "R13", "if": ["tanaman tumbuh kerdil"], "then": "daun menggulung"},
    {"id": "R14", "if": ["muncul gejala water soak"], "then": "tanaman jahe terkena penyakit layu bakteri"}
]

print("=== RULE BASE ===")
for r in rules:
    print(f"{r['id']} : IF {r['if']} THEN {r['then']}")

=== RULE BASE ===
R1 : IF ['tanaman jahe terkena penyakit layu bakteri'] THEN R. solanacearum
R2 : IF ['batang membusuk', 'terlihat luka pada rimpang'] THEN Nematoda
R3 : IF ['muncul eksudat di rimpang'] THEN muncul gejala water soak
R4 : IF ['batang membusuk'] THEN batang terkelupas
R5 : IF ['daun jahe menguning'] THEN daun membusuk
R6 : IF ['daun membusuk', 'batang membusuk'] THEN akar membusuk
R7 : IF ['batang terkelupas'] THEN akar terkelupas
R8 : IF ['daun jahe menguning'] THEN tanaman tumbuh kerdil
R9 : IF ['akar membusuk'] THEN muncul eksudat di batang
R10 : IF ['daun membusuk'] THEN batang membusuk
R11 : IF ['akar terkelupas'] THEN muncul luka di rimpang
R12 : IF ['muncul eksudat di batang'] THEN muncul eksudat di rimpang
R13 : IF ['tanaman tumbuh kerdil'] THEN daun menggulung
R14 : IF ['muncul gejala water soak'] THEN tanaman jahe terkena penyakit layu bakteri


### **2. Inisialisasi Fakta Awal**

In [3]:
facts = set([
    "daun jahe menguning",
    "muncul eksudat di batang"
])

print("=== FAKTA AWAL ===")
for f in facts:
    print("-", f)

=== FAKTA AWAL ===
- daun jahe menguning
- muncul eksudat di batang


### **3. Cek rule yang bisa di eksekusi (preview iterasi awal)**

In [4]:
print("=== RULE YANG AKTIF PADA ITERASI AWAL ===")

for rule in rules:
    if all(cond in facts for cond in rule["if"]):
        print(f"{rule['id']} dapat dieksekusi → menghasilkan: {rule['then']}")

=== RULE YANG AKTIF PADA ITERASI AWAL ===
R5 dapat dieksekusi → menghasilkan: daun membusuk
R8 dapat dieksekusi → menghasilkan: tanaman tumbuh kerdil
R12 dapat dieksekusi → menghasilkan: muncul eksudat di rimpang


ini menunjukkan patern matching sebelum inferensi berjalan

### **4. Forward Chaining**

In [5]:
def forward_chaining_verbose(rules, facts):
    facts = set(facts)
    iteration = 1
    
    print("\n=== MULAI FORWARD CHAINING ===\n")
    
    while True:
        new_fact_found = False
        print(f"\n--- ITERASI {iteration} ---")
        
        for rule in rules:
            if all(cond in facts for cond in rule["if"]):
                
                if rule["then"] not in facts:
                    print(f"[FIRE] {rule['id']}")
                    print(f"Premis terpenuhi: {rule['if']}")
                    print(f"Menambahkan fakta baru: {rule['then']}\n")
                    
                    facts.add(rule["then"])
                    new_fact_found = True
        
        print("Fakta saat ini:")
        for f in facts:
            print("•", f)
        
        if not new_fact_found:
            print("\nTidak ada rule yang dapat dieksekusi lagi.")
            break
        
        iteration += 1
    
    print("\n=== INFERENSI SELESAI ===")
    return facts

### **5. Eksekusi Forward Chaining dan Trace**

In [6]:
final_facts = forward_chaining_verbose(rules, facts)


=== MULAI FORWARD CHAINING ===


--- ITERASI 1 ---
[FIRE] R5
Premis terpenuhi: ['daun jahe menguning']
Menambahkan fakta baru: daun membusuk

[FIRE] R8
Premis terpenuhi: ['daun jahe menguning']
Menambahkan fakta baru: tanaman tumbuh kerdil

[FIRE] R10
Premis terpenuhi: ['daun membusuk']
Menambahkan fakta baru: batang membusuk

[FIRE] R12
Premis terpenuhi: ['muncul eksudat di batang']
Menambahkan fakta baru: muncul eksudat di rimpang

[FIRE] R13
Premis terpenuhi: ['tanaman tumbuh kerdil']
Menambahkan fakta baru: daun menggulung

Fakta saat ini:
• daun membusuk
• batang membusuk
• tanaman tumbuh kerdil
• muncul eksudat di rimpang
• muncul eksudat di batang
• daun jahe menguning
• daun menggulung

--- ITERASI 2 ---
[FIRE] R3
Premis terpenuhi: ['muncul eksudat di rimpang']
Menambahkan fakta baru: muncul gejala water soak

[FIRE] R4
Premis terpenuhi: ['batang membusuk']
Menambahkan fakta baru: batang terkelupas

[FIRE] R6
Premis terpenuhi: ['daun membusuk', 'batang membusuk']
Menambahkan f

### **6. Reasoning Path Extraction**

In [7]:
print("\n=== JALUR INFERENSI KUNCI ===")

key_path = [
    "daun jahe menguning",
    "daun membusuk",
    "batang membusuk",
    "akar membusuk",
    "muncul eksudat di batang",
    "muncul eksudat di rimpang",
    "muncul gejala water soak",
    "tanaman jahe terkena penyakit layu bakteri",
    "R. solanacearum"
]

for step in key_path:
    if step in final_facts:
        print("✔", step)


=== JALUR INFERENSI KUNCI ===
✔ daun jahe menguning
✔ daun membusuk
✔ batang membusuk
✔ akar membusuk
✔ muncul eksudat di batang
✔ muncul eksudat di rimpang
✔ muncul gejala water soak
✔ tanaman jahe terkena penyakit layu bakteri
✔ R. solanacearum


### **7. Diagnosis Akhir**

In [8]:
print("\n=== HASIL DIAGNOSIS ===")

if "R. solanacearum" in final_facts:
    print("Tanaman terdiagnosis terkena bakteri Ralstonia solanacearum")
elif "Nematoda" in final_facts:
    print("Tanaman terdiagnosis terkena Nematoda")
else:
    print("Diagnosis tidak ditemukan")


=== HASIL DIAGNOSIS ===
Tanaman terdiagnosis terkena bakteri Ralstonia solanacearum
